<a href="https://colab.research.google.com/github/gflamiano/SJU-CSC-672-ML/blob/main/HW7/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 7 Analysis

#### 1. Why is feature scaling essential prior to executing K-Means, DBSCAN, or HDBSCAN on the Wine dataset? Specifically, how would unscaled features distort distance calculations relative to other features? Provide specific examples.

All three algorithms depend on distances between samples (K-Means measures distance to a centroid, DBSCAN uses `eps`, and HDBSCAN builds its tree from distances). Euclidean distance adds up the squared difference of every feature, so a feature measured in large numbers takes over the sum. The Wine features are on very different scales. In our `describe()` output, `proline` runs from 278 to 1680 (standard deviation about 315) and `magnesium` from 70 to 162 (standard deviation about 14), while `hue` runs from 0.48 to 1.71 (standard deviation about 0.23) and `nonflavanoid_phenols` from 0.13 to 0.66.

Here is a concrete example. Take two wines that differ by one standard deviation in both `proline` and `hue`. The `proline` difference adds about 315² ≈ 99,000 to the squared distance, while the `hue` difference adds only about 0.23² ≈ 0.05. That makes `proline` roughly two million times more influential. `magnesium` also dominates `hue`, by a factor of about 4,000. Even the largest possible `hue` difference (about 1.2) adds only about 1.5, which is tiny next to a modest `proline` difference. Without scaling, the clusters would be based almost entirely on `proline` and `magnesium`, and the other 11 features would barely count. The same problem affects the density methods, since `eps` would effectively measure only `proline` distance. After standardization, every feature has a mean of 0 and a standard deviation of 1, so each one contributes equally, and a single `eps` value (such as 2.5) means the same thing in every direction.



---



#### 2. How did the Elbow Method and Silhouette Analysis in Task 2 reflect the natural structure of the Wine dataset? Did both evaluation metrics align with the true number of wine cultivars (k=3)? Explain any discrepancies observed.

Both metrics suggested k=3, which matches the three real wine cultivars. The elbow curve falls steeply from k=1 (inertia about 2,300) to k=3 (about 1,280), then flattens, so adding a fourth cluster gains very little. The silhouette score peaks at k=3 with a value of about 0.285, compared with about 0.26 at both k=2 and k=4. K-Means with k=3 also predicted the true cultivars with an accuracy of 0.972 in Task 8, which confirms that these three clusters match the real groups.

There are some discrepancies, though. The elbow is not sharp. The curve keeps falling slowly after k=3, so the bend is only clear because we picked the point farthest from the straight line between the curve's endpoints, and judging it by eye is somewhat subjective. The silhouette scores are also low overall. A score of 0.285 is well below the 0.5 usually considered a strong structure, and k=2 and k=4 score only about 0.025 lower, so the preference for k=3 is small. There is also a smaller bump at k=6 (about 0.24), which has no meaning for the cultivars. Together these show that the three cultivars are real but overlap, with no clean gaps between them. Both metrics measure compactness and separation in the 13 chemical features, not agreement with the cultivar labels, so it is a good result that they both landed on 3.



---



#### 3. Compare the clustering behavior and cluster assignments of K-Means against DBSCAN and HDBSCAN on the 13-dimensional Wine feature space. Why did DBSCAN and HDBSCAN classify specific wine samples as noise, while K-Means assigned every sample to a cluster?

K-Means splits the space into 3 regions around centroids and puts every sample in the region of its nearest centroid. It has no idea of an outlier, so every sample gets a cluster, however far away it is. In Task 4, Sample 121 was assigned to cluster 2 even though its distances to the three centroids were 6.33, 7.48, and 6.16. It is far from all of them and nearly equally far from two, but it was still forced into a cluster.

DBSCAN and HDBSCAN define clusters by density instead, so they can reject samples. DBSCAN with `eps=2.5` and `min_samples=5` found only 1 cluster and labeled 24 samples as noise. HDBSCAN with `min_cluster_size=5` found 2 clusters and labeled 58 samples as noise, about a third of the data. A sample is noise when it lacks enough close neighbors to belong to a dense region. The Wine samples are spread out across 13 dimensions, and in high dimensions samples are far from each other, so the data is sparse. The dendrogram in Task 7 shows this too: most samples merge gradually into one large group, with no clear gaps between the cultivars, and a small number of samples join only at large distances. This is also why neither density method recovered three clusters. The cultivars are not separated by low-density gaps, so density-based methods struggle to split them, while K-Means succeeds because it only needs the groups to be roughly centered around three points.



---



#### 4. Contrast the operational mechanisms of standard DBSCAN and HDBSCAN. How does HDBSCAN's hierarchical, stability-based approach overcome DBSCAN's vulnerability to a single global `eps` parameter when clustering chemical datasets with varying local densities?

DBSCAN uses two settings, `eps` (a neighborhood radius) and `min_samples`. It marks a sample as a core point if it has at least `min_samples` neighbors within `eps`, joins connected core points into clusters, and labels everything else as noise. Because there is one `eps` for the whole dataset, it can only find clusters at one density level. If one group is dense and another is spread out, no single `eps` works for both: a small value breaks the sparse group into noise, and a large value merges the dense groups together. Our results show this sensitivity. With `min_samples=5`, `eps=1.5` labeled all 178 samples as noise, `eps=2.0` gave 5 clusters with 85 noise points, and `eps=2.5` gave 1 cluster with 24 noise points. No setting gave 3 clusters.

HDBSCAN removes `eps`. It builds a single-linkage hierarchy using a distance that pushes sparse samples further away, which is effectively the same as running DBSCAN at every possible `eps` at once. It then simplifies the tree by keeping only splits where both sides have at least `min_cluster_size` samples. Each remaining cluster gets a stability score, based on how long it survives and how many samples it holds as the distance changes. HDBSCAN keeps the most stable clusters, and it can do this separately for each branch, so a dense cluster and a sparse cluster can each be kept at their own density level. The only setting that matters is `min_cluster_size`. In our case, HDBSCAN did give a result without any tuning of a distance threshold, but it found 2 clusters with 58 noise points, not the 3 cultivars. Its advantage is flexibility with varying densities, and it cannot create density gaps that the data does not have.



---



#### 5. Based on your comparative classification experiment in Task 8, how effectively did the cluster assignments from K-Means, DBSCAN, and HDBSCAN enable a downstream classifier to predict the true ground-truth wine cultivars (Class 0-2)? Discuss the limitations of mapping unsupervised clusters to supervised class labels when dealing with noise and density-based groupings.

K-Means worked very well. Its cluster labels gave a test accuracy of 0.972 (35 of 36 test samples), so its three clusters line up almost exactly with the three cultivars. HDBSCAN was moderately effective, with an accuracy of 0.750 (27 of 36). DBSCAN was the weakest, with an accuracy of 0.389 (14 of 36), which is about the same as always guessing the most common cultivar (71 of the 178 wines, about 40%).

These results show the limits of using unsupervised clusters as class predictors. First, the number of clusters limits the accuracy. DBSCAN found 1 cluster, so its labels take only two values (cluster 0 and noise), and two values cannot separate three cultivars. HDBSCAN found 2 clusters, so at least one cultivar must share a cluster with another. Second, noise is not a class. All noise points share one label, but they are samples from different cultivars that just happen to be isolated, so that label has little information. HDBSCAN had 58 noise points, a third of the data, which explains much of its lower accuracy. Third, density-based groups follow density, not chemistry. Two cultivars that overlap in the feature space merge into one dense region, and a cultivar can be split between a dense core and sparse edges that become noise. Fourth, cluster numbers are arbitrary names, so the classifier has to learn the mapping from clusters to cultivars, and it can only do this well if each cluster mostly contains one cultivar. Finally, the Silhouette Scores can be misleading. HDBSCAN had the highest score (0.352) compared with K-Means (0.285), but its score ignores the 58 noise points, and DBSCAN's score could not be computed at all (N/A), because it found only 1 cluster. The test set is also small (36 samples), so each sample is worth about 2.8 percentage points of accuracy.



---

